# Gutenberg domain: why TF-IDF fails, and features for `human` / `fusion`
Competition: [ai-generated-text-detection-hw3-ucu-ai-2026](https://www.kaggle.com/competitions/ai-generated-text-detection-hw3-ucu-ai-2026). This notebook looks only at `domain == "gutenberg"` (7,344 train rows, 1,845 test rows).

**Findings** (numbers from a local run; this notebook recomputes them):
- **Extreme imbalance**: ~91% of Gutenberg train rows are `ai`. Only 247 are `human` and 378 are `fusion`, across both languages; German has 31 human and 37 fusion. A generic model learns "Gutenberg means ai", and plain TF-IDF + LinearSVC reaches only ~0.51 macro-F1 here (F1 human ~0.18, fusion ~0.36).
- **What the classes look like**:
  - `human` texts are real Project Gutenberg books: title pages, publisher and copyright lines, *Transcriber's Notes*, odd typography (`·`, `½`, spaced punctuation ` ,`), ALL-CAPS headings, and very uneven sentence lengths.
  - `ai` texts are generated fiction: fluent, even rhythm, AI vocabulary (*testament, whisper, tapestry, symphony, etched…*), and sometimes LLM meta-replies ("Please provide the initial text…").
  - `fusion` texts are a human Gutenberg text with **AI-written passages inserted**, anywhere in the text, not only at the end.
- **Human vs ai is easy, fusion vs human is hard.** Whole-text features separate human from ai (AUC ~0.9), but fusion vs human only reaches ~0.75, because a fusion text is mostly human.
- **The key feature is window-level AI probability.** Split each text into ~150-word windows and score each window with a small human-vs-ai classifier. Then summarise per text: max, min, mean, spread, and share of AI windows. A fusion text has **both** human-looking and AI-looking windows. This raises fusion-vs-human AUC to ~0.88 and fusion-vs-ai to ~0.99.
- **Result**: Gutenberg-specific features + window features + TF-IDF scores, stacked in a gradient-boosting model with class weights, reach **~0.94 macro-F1** out-of-fold (human ~0.91, fusion ~0.93).

**Running**: CPU only, no internet needed, roughly 10–20 min. It writes `gutenberg_features.csv` (all features for Gutenberg train + test rows) to `/kaggle/working`, so the features can be reused in the main models.

## Setup & load

In [ ]:
import os, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TEXT, LANG, DOMAIN, LABEL = "text", "lang", "domain", "label"
CLASSES = ["human", "ai", "fusion"]
LANGS = ["en-EN", "de-DE"]
COMP = "ai-generated-text-detection-hw3-ucu-ai-2026"

def find_data_dir():
    for root in [Path("/kaggle/input"), Path("data"), Path("../data")]:
        if root.exists():
            hits = sorted(root.rglob("train.csv"))
            hits = [h for h in hits if COMP in str(h)] or hits
            if hits:
                return hits[0].parent
    raise FileNotFoundError("train.csv not found: add the competition data as an input")

DATA_DIR = find_data_dir()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
g = train[train[DOMAIN] == "gutenberg"].reset_index(drop=True)
gt = test[test[DOMAIN] == "gutenberg"].reset_index(drop=True)
for df in (g, gt):
    df[TEXT] = df[TEXT].fillna("").astype(str)
del train, test

y = g[LABEL].values
print(f"gutenberg train: {len(g):,} | test: {len(gt):,}")
display(pd.crosstab(g[LANG], g[LABEL], margins=True)[CLASSES + ["All"]])
display(pd.crosstab(g[LANG], g[LABEL], normalize="index")[CLASSES].round(3))
print("test rows per language:", gt[LANG].value_counts().to_dict())

## What the classes look like
Start and end of random examples. Note the publishing metadata and transcriber's notes in `human`, the generated fiction in `ai`, and human text with AI-style passages in `fusion`.

In [ ]:
for lab in CLASSES:
    print(f"\n==================== {lab}")
    for _, r in g[g[LABEL] == lab].sample(3, random_state=1).iterrows():
        t = r[TEXT]
        print(f"--- [{r[LANG]}] {len(t):,} chars\n  START: {t[:350]!r}\n  END:   {t[-250:]!r}")

g["n_chars"] = g[TEXT].str.len()
print("\nText length (chars) by language x label:")
display(g.groupby([LANG, LABEL])["n_chars"].describe()[["count", "25%", "50%", "75%", "max"]].round(0))

## Why TF-IDF fails here
Out-of-fold (5-fold, stratified) TF-IDF word 1–2-grams + char 3–5-grams with LinearSVC, trained on Gutenberg only. Even with the whole training budget spent on this domain, most `human` texts are predicted `fusion` and vice versa, and many fall into `ai`. Both classes are rare, and both are mostly *human-written* text, so their n-grams look alike.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(np.zeros(len(y)), y))

def tfidf_fit(texts):
    wv = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=300_000)
    cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=3, max_features=300_000)
    X = sp.hstack([wv.fit_transform(texts), cv.fit_transform([t[:2000] for t in texts])]).tocsr()
    return (wv, cv), X

def tfidf_transform(vecs, texts):
    wv, cv = vecs
    return sp.hstack([wv.transform(texts), cv.transform([t[:2000] for t in texts])]).tocsr()

def report(name, pred):
    f = f1_score(y, pred, average=None, labels=CLASSES)
    return {"model": name, "macro_f1": f1_score(y, pred, average="macro"), "f1_human": f[0], "f1_ai": f[1], "f1_fusion": f[2]}

results, preds = [], {}
for name, cw in [("TF-IDF + LinearSVC", None), ("TF-IDF + LinearSVC, class-balanced", "balanced")]:
    pred = np.empty(len(y), dtype=object)
    for tr, te in FOLDS:
        vecs, Xtr = tfidf_fit(g[TEXT].values[tr].tolist())
        pred[te] = LinearSVC(C=0.3, class_weight=cw).fit(Xtr, y[tr]).predict(tfidf_transform(vecs, g[TEXT].values[te].tolist()))
    preds[name] = pred
    results.append(report(name, pred))
    print(results[-1])

def plot_cm(pred, title, ax):
    cm = confusion_matrix(y, pred, labels=CLASSES)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(title); ax.set_xlabel("predicted"); ax.set_ylabel("true")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, name in zip(axes, preds):
    plot_cm(preds[name], name, ax)
plt.tight_layout(); plt.show()

## Gutenberg-specific handcrafted features
Candidate features, grouped:
- **Publishing metadata** (strongly `human`): transcriber's notes, Project Gutenberg / proofreading credits, copyright/printed lines, edition, chapter, contents, footnote, illustration. The same markers are counted in the first 30% and last 30% of the text. A German list is included (*Verlag, Auflage, Druckfehler…*).
- **Layout / typography**: ALL-CAPS words (title pages), digits, special symbols (`· ½ † § ¶ £`), `_italics_` markup, brackets, a **space before punctuation** (` ,` ` .`, an OCR/transcription artefact), em dashes, curly vs straight quotes, Roman numerals.
- **Rhythm**: sentence-length std and burstiness (human prose is much more uneven), average sentence and word length, type-token ratio, semicolons/colons, whether the text ends with punctuation (German human texts often stop mid-sentence).
- **AI vocabulary**: words typical of generated fiction (*testament, whisper, tapestry, symphony, etched, vibrant, unwavering…*), in the whole text, the first 30% and the last 30%. LLM meta-replies ("please provide", "word count"). A German list is included.
- **Archaic spelling**: *to-day, 'tis, thee, hath, connexion*, and German *Thür, Theil, seyn, daß…*. This marks older human text.
- **Length**: human and fusion texts are much longer than ai texts.

In [ ]:
WORD = re.compile(r"\w+")
GUT = ["transcriber", "project gutenberg", "gutenberg", "proofread", "pgdp", "distributed proofreading", "produced by",
       "copyright", "printed", "publisher", "published", "edition", "illustrat", "chapter", "contents", "footnote",
       "[illustration", "www.", "http", "etext", "ebook", "e-text", "obvious", "typographical", "errata", "retained",
       "spelling", "original text", "hyphenation", "inconsisten", "page", "vol.", "price"]
DE_GUT = ["anmerkung", "transkription", "druckfehler", "verlag", "auflage", "inhalt", "kapitel", "gedruckt",
          "herausgegeben", "fußnote", "seite"]
ARCH = ["to-day", "to-morrow", "to-night", "’tis", "'tis", " thee ", " thou ", " thy ", " hath ", " doth ", " shew",
        "connexion", " ye ", "whilst", "o’er", "e’en", "&c", " viz", " ere "]
DE_ARCH = ["thür", "theil", "seyn", "giebt", "thun", "daß", "muth", "werth", "noth", "blos", "gieng"]
AIW = ["testament", "whisper", "tapestry", "symphony", "etched", "vibrant", "embrace", "unwavering", "realm", "beacon",
       "echo", "resilience", "intricate", "a sense of", "couldn't help but", "couldn’t help but", "palpable", "flicker",
       "linger", "shimmer", "weight of", "heart pounding", "breath", "determination", "newfound", "profound", "delve",
       "navigat", "solace", "ethereal", "serene", "kaleidoscope", "danced", "cacophony", "sanctuary", "steeled", "resolve"]
DE_AI = ["flüster", "zeugnis", "symphonie", "geheimnisvoll", "unerschütterlich", "hauch", "entschlossen", "glanz",
         "sehnsucht", "atem"]
META = ["please provide", "word count", "here is", "here's", "i hope", "as an ai", "certainly", "i cannot", "rewrite", "expand upon"]
SPECIAL = "·½¼¾†‡§¶°£"

def cnt(low, words):
    return sum(low.count(w) for w in words)

def part(t, a, b):
    n = len(t)
    return t[int(n * a):int(n * b)]

def gutenberg_features(t):
    low = t.lower()
    words = WORD.findall(t)
    nw, n = max(len(words), 1), max(len(t), 1)
    first, last = part(low, 0, .3), part(low, .7, 1)
    nfw, nlw = max(len(WORD.findall(first)), 1), max(len(WORD.findall(last)), 1)
    sents = [s for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]
    sl = np.array([len(WORD.findall(s)) for s in sents] or [0])
    caps = sum(1 for w in words if len(w) >= 2 and w.isupper())
    first_words = WORD.findall(part(t, 0, .3))
    return {
        # metadata
        "gut_markers": cnt(low, GUT), "gut_markers_first30": cnt(first, GUT), "gut_markers_last30": cnt(last, GUT),
        "transcriber_note": int("transcriber" in low), "copyright_or_printed": int("copyright" in low or "printed" in low),
        "de_gut_markers": cnt(low, DE_GUT),
        # layout / typography
        "caps_word_ratio": caps / nw,
        "caps_first30": sum(1 for w in first_words if len(w) >= 2 and w.isupper()) / max(len(first_words), 1),
        "digit_ratio": sum(c.isdigit() for c in t) / n,
        "special_symbols": sum(t.count(c) for c in SPECIAL) / n * 1000,
        "underscore_markup": t.count("_") / n * 1000,
        "brackets": (t.count("[") + t.count("{")) / n * 1000,
        "space_before_punct": len(re.findall(r"\s[,.;:]", t)) / n * 1000,
        "em_dash": t.count("—") / n * 1000,
        "curly_quotes": sum(t.count(c) for c in "“”‘’") / n * 1000,
        "straight_quotes": (t.count('"') + t.count("'")) / n * 1000,
        "roman_numerals": len(re.findall(r"\b[IVXLC]{2,}\b", t)) / nw * 1000,
        "starts_with_caps_block": int(bool(re.match(r"^[^a-z]{25}", t))),
        # rhythm
        "avg_sent_len": sl.mean(), "std_sent_len": sl.std(), "burstiness": sl.std() / (sl.mean() + 1e-9),
        "avg_word_len": np.mean([len(w) for w in words]) if words else 0.0,
        "ttr_first1000w": len(set(w.lower() for w in words[:1000])) / max(min(len(words), 1000), 1),
        "semicolon": t.count(";") / nw * 100, "colon": t.count(":") / nw * 100, "excl": t.count("!") / nw * 100,
        "ends_with_punct": int(t.rstrip()[-1:] in ".!?\"”’'"),
        # AI vocabulary
        "ai_words": cnt(low, AIW) / nw * 1000,
        "ai_words_first30": cnt(first, AIW) / nfw * 1000, "ai_words_last30": cnt(last, AIW) / nlw * 1000,
        "de_ai_words": cnt(low, DE_AI) / nw * 1000,
        "llm_meta": cnt(low, META),
        # archaic spelling
        "archaic": cnt(" " + low + " ", ARCH) / nw * 1000, "de_archaic": cnt(low, DE_ARCH) / nw * 1000,
        # length
        "n_chars": len(t), "n_words": len(words),
    }

t0 = time.time()
F = pd.DataFrame([gutenberg_features(t) for t in g[TEXT]])
F_test = pd.DataFrame([gutenberg_features(t) for t in gt[TEXT]])
F["is_de"] = (g[LANG] == "de-DE").astype(int).values
F_test["is_de"] = (gt[LANG] == "de-DE").astype(int).values
print(f"{F.shape[1]} features in {time.time() - t0:.0f}s")

### Signal per feature (ROC AUC, per language)
0.5 means no signal; values far from 0.5 in either direction mean signal. Four questions: is it human (vs rest), is it fusion (vs rest), **fusion vs human** (the hard pair), and fusion vs ai.

In [ ]:
def auc_table(F, y, mask):
    yy = y[mask]
    rows = {}
    for c in F.columns:
        x = F[c].values[mask]
        def auc(sel, pos):
            return roc_auc_score(yy[sel] == pos, x[sel]) if len(set(x[sel])) > 1 and len(set(yy[sel] == pos)) > 1 else 0.5
        allm = np.ones(len(yy), bool)
        rows[c] = {"human_vs_rest": auc(allm, "human"), "fusion_vs_rest": auc(allm, "fusion"),
                   "fusion_vs_human": auc(np.isin(yy, ["human", "fusion"]), "fusion"),
                   "fusion_vs_ai": auc(yy != "human", "fusion"),
                   **{f"median_{k}": np.median(x[yy == k]) for k in CLASSES}}
    R = pd.DataFrame(rows).T
    R["strength"] = (R[["human_vs_rest", "fusion_vs_rest", "fusion_vs_human", "fusion_vs_ai"]] - 0.5).abs().max(axis=1)
    return R.sort_values("strength", ascending=False)

auc_cols = ["human_vs_rest", "fusion_vs_rest", "fusion_vs_human", "fusion_vs_ai"]
AUC = {}
for lang in LANGS:
    AUC[lang] = auc_table(F.drop(columns="is_de"), y, (g[LANG] == lang).values)
    print(f"===== {lang}")
    display(AUC[lang].round(3).style.background_gradient(subset=auc_cols, cmap="coolwarm", vmin=0, vmax=1))

In [ ]:
top = AUC["en-EN"].index[:12].tolist()
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, c in zip(axes.flat, top):
    d = pd.DataFrame({"v": F[c], LABEL: y, LANG: g[LANG]})
    lo, hi = d["v"].quantile([0.01, 0.99])
    sns.boxplot(data=d.assign(v=d["v"].clip(lo, hi)), x=LABEL, y="v", hue=LANG, order=CLASSES, hue_order=LANGS,
                ax=ax, showfliers=False, legend=(c == top[0]))
    ax.set_title(c); ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle("Top handcrafted features (by AUC strength, en-EN)"); plt.tight_layout(); plt.show()

## Window-level AI probability: the fusion detector
A fusion text is mostly human, with AI passages inserted. Whole-text averages dilute those passages, so we score **windows** instead:
1. Split each text into windows of `WIN_WORDS` words.
2. Train a small char 2–4-gram TF-IDF + logistic regression on windows from **pure** `human` vs pure `ai` texts.
3. For each text, summarise its window scores: max, min, mean, std, range, share of windows above/below 0.5, whether it has both a clearly-AI and a clearly-human window, and last-minus-first.

To avoid leakage, window scores for a text always come from a model that did not see that text (out-of-fold).

In [ ]:
WIN_WORDS = 150

def windows(t, size=WIN_WORDS):
    w = t.split()
    return [" ".join(w[i:i + size]) for i in range(0, max(len(w), 1), size)] or [t]

WIN = [windows(t) for t in g[TEXT]]
WIN_TEST = [windows(t) for t in gt[TEXT]]

def fit_window_model(doc_idx):
    Xw, yw = [], []
    for i in doc_idx:
        if y[i] == "human":
            Xw += WIN[i]; yw += [0] * len(WIN[i])
        elif y[i] == "ai":
            Xw += WIN[i][:4]; yw += [1] * len(WIN[i][:4])     # cap ai windows per text: ~27x more ai texts
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True, min_df=3, max_features=200_000)
    clf = LogisticRegression(C=2, max_iter=2000, class_weight="balanced").fit(vec.fit_transform(Xw), yw)
    return vec, clf

def score_windows(model, win_lists):
    vec, clf = model
    flat = [w for ws in win_lists for w in ws]
    p = clf.predict_proba(vec.transform(flat))[:, 1]
    out, k = [], 0
    for ws in win_lists:
        out.append(p[k:k + len(ws)]); k += len(ws)
    return out

def window_summary(scores):
    return pd.DataFrame({
        "win_ai_max": [s.max() for s in scores], "win_ai_min": [s.min() for s in scores],
        "win_ai_mean": [s.mean() for s in scores], "win_ai_std": [s.std() for s in scores],
        "win_ai_range": [s.max() - s.min() for s in scores],
        "win_frac_ai": [(s > .5).mean() for s in scores], "win_frac_human": [(s < .5).mean() for s in scores],
        "win_has_both": [float((s > .7).any() and (s < .3).any()) for s in scores],
        "win_last_minus_first": [s[-1] - s[0] for s in scores], "win_n": [len(s) for s in scores]})

def oof_window_scores(doc_idx, n_splits=5):
    """Out-of-fold window scores for the docs in doc_idx (each scored by a model trained on the other docs)."""
    doc_idx = np.asarray(doc_idx)
    scores = [None] * len(doc_idx)
    inner = StratifiedKFold(n_splits, shuffle=True, random_state=RANDOM_STATE)
    for tr, te in inner.split(doc_idx, y[doc_idx]):
        sc = score_windows(fit_window_model(doc_idx[tr]), [WIN[i] for i in doc_idx[te]])
        for j, s in zip(te, sc):
            scores[j] = s
    return scores

t0 = time.time()
win_scores = oof_window_scores(np.arange(len(g)))
W = window_summary(win_scores)
print(f"window features in {time.time() - t0:.0f}s")
for lang in LANGS:
    print(f"===== {lang}")
    display(auc_table(W, y, (g[LANG] == lang).values).round(3)
            .style.background_gradient(subset=auc_cols, cmap="coolwarm", vmin=0, vmax=1))

### Where are the AI passages inside fusion texts?
Mean window AI probability by relative position in the text. A flat profile means the AI passages are inserted **anywhere**, not only at the end. That matters for the transformer, too: head + tail truncation can miss an AI passage in the middle, while sliding-window chunks cover it.

In [ ]:
pos = pd.DataFrame([(i / (len(s) - 1), p, lab) for s, lab in zip(win_scores, y) if len(s) >= 3
                    for i, p in enumerate(s)], columns=["rel_pos", "p_ai", LABEL])
pos["bin"] = pd.cut(pos["rel_pos"], np.linspace(-0.001, 1, 11))
prof = pos.groupby([LABEL, "bin"], observed=True)["p_ai"].mean().unstack(0)[CLASSES]
prof.index = [f"{iv.right:.1f}" for iv in prof.index]
prof.plot(marker="o", figsize=(9, 4), title="Mean window AI probability by position in the text")
plt.xlabel("relative position (end of bin)"); plt.ylabel("P(ai) of window"); plt.ylim(0, 1); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(6, 1, figsize=(12, 6))
ex = g.index[(y == "fusion") & (W["win_n"].values >= 8)][:6]
for ax, i in zip(axes, ex):
    sns.heatmap(win_scores[i][None, :], ax=ax, cmap="RdBu_r", vmin=0, vmax=1, cbar=False, xticklabels=False, yticklabels=False)
    ax.set_ylabel(f"#{i}", rotation=0, labelpad=20)
axes[0].set_title("Window AI probability along 6 fusion texts (blue = human-like, red = AI-like)")
plt.tight_layout(); plt.show()

### A train-only artefact: shared prefixes
About 29% of `fusion` train texts share their first 300 characters with another fusion text: the same human source with different AI insertions. No pure `ai` text and almost no `human` text does. That would be a strong feature, but **no test text shares a prefix with any train or test text**, so it cannot help on test. It is reported here and left out of the models.

In [ ]:
pref = pd.concat([g[TEXT].str[:300], gt[TEXT].str[:300]]).value_counts()
print("train: share sharing a 300-char prefix with another text, by label:",
      (g[TEXT].str[:300].map(pref) > 1).groupby(y).mean().round(3).to_dict())
print("test: share sharing a 300-char prefix with another text:", round((gt[TEXT].str[:300].map(pref) > 1).mean(), 3))

## How much do the features help? (5-fold out-of-fold, Gutenberg only)
All models use a gradient-boosting classifier with balanced class weights, except the TF-IDF rows. Window features and TF-IDF scores are computed **inside** each outer fold (nested out-of-fold), so no score is optimistic through leakage.

In [ ]:
hgb = lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced",
                                             random_state=RANDOM_STATE)

def tfidf_scores(tr_idx, te_idx):
    """TF-IDF LinearSVC decision scores: out-of-fold for tr_idx, from a model on all tr_idx for te_idx."""
    tr_idx = np.asarray(tr_idx)
    s_tr = np.zeros((len(tr_idx), 3))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(tr_idx, y[tr_idx]):
        vecs, X = tfidf_fit(g[TEXT].values[tr_idx[a]].tolist())
        m = LinearSVC(C=0.3, class_weight="balanced").fit(X, y[tr_idx[a]])
        s_tr[b] = m.decision_function(tfidf_transform(vecs, g[TEXT].values[tr_idx[b]].tolist()))
    vecs, X = tfidf_fit(g[TEXT].values[tr_idx].tolist())
    m = LinearSVC(C=0.3, class_weight="balanced").fit(X, y[tr_idx])
    return s_tr, m.decision_function(tfidf_transform(vecs, g[TEXT].values[te_idx].tolist()))

def window_feats(tr_idx, te_idx):
    w_tr = window_summary(oof_window_scores(tr_idx)).values
    w_te = window_summary(score_windows(fit_window_model(tr_idx), [WIN[i] for i in te_idx])).values
    return w_tr, w_te

configs = {"handcrafted": (False, False), "handcrafted + window": (True, False),
           "handcrafted + window + TF-IDF scores (stack)": (True, True)}
oof = {k: np.empty(len(y), dtype=object) for k in configs}
t0 = time.time()
for f_i, (tr, te) in enumerate(FOLDS):
    w_tr, w_te = window_feats(tr, te)
    s_tr, s_te = tfidf_scores(tr, te)
    for name, (use_w, use_s) in configs.items():
        Xtr = [F.values[tr]] + ([w_tr] if use_w else []) + ([s_tr] if use_s else [])
        Xte = [F.values[te]] + ([w_te] if use_w else []) + ([s_te] if use_s else [])
        oof[name][te] = hgb().fit(np.hstack(Xtr), y[tr]).predict(np.hstack(Xte))
    print(f"fold {f_i + 1}/5 done ({time.time() - t0:.0f}s)", flush=True)

for name in configs:
    results.append(report(name, oof[name]))
res = pd.DataFrame(results).set_index("model")
display(res.style.background_gradient(cmap="Greens", vmin=0, vmax=1).format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_cm(preds["TF-IDF + LinearSVC"], "TF-IDF + LinearSVC", axes[0])
best_name = res.index[-1]
plot_cm(oof[best_name], best_name, axes[1])
plt.tight_layout(); plt.show()

for lang in LANGS:
    m = (g[LANG] == lang).values
    print(f"{lang}: TF-IDF macro-F1={f1_score(y[m], preds['TF-IDF + LinearSVC'][m], average='macro'):.3f} | "
          f"stack macro-F1={f1_score(y[m], oof[best_name][m], average='macro'):.3f}")

### Which feature groups matter? (drop one group at a time, handcrafted + window model)

In [ ]:
GROUPS = {
    "metadata": ["gut_markers", "gut_markers_first30", "gut_markers_last30", "transcriber_note", "copyright_or_printed", "de_gut_markers"],
    "layout/typography": ["caps_word_ratio", "caps_first30", "digit_ratio", "special_symbols", "underscore_markup", "brackets",
                          "space_before_punct", "em_dash", "curly_quotes", "straight_quotes", "roman_numerals", "starts_with_caps_block"],
    "rhythm": ["avg_sent_len", "std_sent_len", "burstiness", "avg_word_len", "ttr_first1000w", "semicolon", "colon", "excl", "ends_with_punct"],
    "ai_vocabulary": ["ai_words", "ai_words_first30", "ai_words_last30", "de_ai_words", "llm_meta"],
    "archaic": ["archaic", "de_archaic"],
    "length": ["n_chars", "n_words"],
}
# window features from the first pass (single-level OOF) are fine for a relative comparison between groups
def cv_macro(X):
    p = np.empty(len(y), dtype=object)
    for tr, te in FOLDS:
        p[te] = hgb().fit(X[tr], y[tr]).predict(X[te])
    return f1_score(y, p, average="macro")

base = cv_macro(np.hstack([F.values, W.values]))
abl = [{"dropped": "(none)", "macro_f1": base, "delta": 0.0},
       {"dropped": "ALL window features", "macro_f1": (s := cv_macro(F.values)), "delta": s - base}]
for gname, cols in GROUPS.items():
    s = cv_macro(np.hstack([F.drop(columns=cols).values, W.values]))
    abl.append({"dropped": gname, "macro_f1": s, "delta": s - base})
display(pd.DataFrame(abl).style.background_gradient(subset=["delta"], cmap="RdYlGn", vmin=-0.05, vmax=0.05).format(precision=3))

## Export features for Gutenberg train + test
For test rows, the window model is trained on **all** Gutenberg train texts. Train rows keep their out-of-fold window scores. The file can be merged into the main pipelines by `id`.

In [ ]:
W_test = window_summary(score_windows(fit_window_model(np.arange(len(g))), WIN_TEST))
out = pd.concat([
    pd.concat([g[["id", LANG, LABEL]].assign(split="train"), F.drop(columns="is_de"), W], axis=1),
    pd.concat([gt[["id", LANG]].assign(split="test", **{LABEL: np.nan}), F_test.drop(columns="is_de"), W_test], axis=1),
], ignore_index=True)
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
out.to_csv(OUT_DIR / "gutenberg_features.csv", index=False)
print(f"saved {(OUT_DIR / 'gutenberg_features.csv').resolve()} {out.shape}")
display(out.groupby(["split", LABEL], dropna=False)[["win_ai_max", "win_ai_min", "win_frac_ai", "gut_markers", "burstiness", "ai_words"]]
        .median().round(3))

## Takeaways for the main models
- **Use window-level AI-probability features.** They are the single strongest signal for `fusion` (it has both human-like and AI-like windows) and for `human` (all windows human-like). They work in any domain where fusion means inserted AI passages, not just Gutenberg.
- **Treat Gutenberg separately, or weight it.** Use class weights, or a separate Gutenberg model: TF-IDF trained on all domains sees Gutenberg as "almost always ai".
- **Useful handcrafted signals**: publishing metadata, ALL-CAPS / title-page layout, space before punctuation, sentence-length burstiness, AI-fiction vocabulary, length.
- **Stacking**: TF-IDF decision scores + handcrafted + window features in a gradient-boosting model gave the best Gutenberg macro-F1.
- **For mmBERT**: AI passages sit anywhere in fusion texts, so prefer sliding-window chunks over head + tail truncation for this domain.
- **Don't use shared-prefix duplicates**: they exist only in train.